# 동기 vs. 비동기 처리

In [1]:
import asyncio
import time
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

questions = [f"주식 투자 팁 {i}번째" for i in range(10)]

# 비동기 병렬 처리
async def invoke_async(question):
    response = await llm.ainvoke(question)
    return response.content

async def invoke_parallel(questions):
    tasks = [invoke_async(q) for q in questions]
    return await asyncio.gather(*tasks)

In [2]:
# 비동기 실행 시간 측정
start = time.perf_counter()
results = await invoke_parallel(questions)
end = time.perf_counter()
print(f"비동기 처리: {end - start:.2f}초")   # 약 5~6초

# 동기 실행 시간 측정
start = time.perf_counter()
for q in questions:
    response = llm.invoke(q)
end = time.perf_counter()
print(f"동기 처리: {end - start:.2f}초")     # 약 40~50초

비동기 처리: 7.20초
동기 처리: 54.03초


# Token 사용량 및 비용

In [ ]:
from langchain_community.callbacks import get_openai_callback
from langchain_core.messages import HumanMessage

with get_openai_callback() as callback:
    message = [HumanMessage(content="서울 광장시장에서 가장 맛있는 길거리 음식 하나만 추천해줘?")]
    # message = [HumanMessage(content="서울 광장시장에서 가장 맛있는 길거리 음식은?")]
    response = llm.invoke(message)
    
print(response.content)
print(f"\n총 토큰: {callback.total_tokens}")
print(f"프롬프트 토큰: {callback.prompt_tokens}")
print(f"응답 토큰: {callback.completion_tokens}")
print(f"비용(USD): ${callback.total_cost:.6f}")

서울 광장시장에서 가장 추천할 만한 길거리 음식은 ***떡볶이***입니다. 매콤달콤한 소스에 쫄깃한 떡과 다양한 재료들이 어우러져 특히 인기 있는 메뉴입니다. 광장시장에서 판매하는 떡볶이는 맛이 다양하고 푸짐하게 나오는 곳이 많으니 꼭 한번 드셔보세요!

총 토큰: 113
프롬프트 토큰: 24
응답 토큰: 89
비용(USD): $0.000057


- (1) [system], [user: 프롬프트#1] ──▶ LLM ──▶ [assistant: 응답#1
- (2) [system], [user: 프롬프트#1], [assistant: 응답#1], [user: 프롬프트#2] ──▶ LLM ──▶ [assistant: 응답#2]
- (3) [system], [user: 프롬프트#1], [assistant: 응답#1], [user: 프롬프트#2], [assistant: 응답#2], [user: 프롬프트#3] ──▶ …

# Conversation history

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")

# 대화 시작
messages = [
    SystemMessage(content="당신은 주식 투자 전문가로 고객의 투자 결정에 도움을 줍니다."),
    HumanMessage(content="삼성전자 주식을 지금 매수해도 될까요?")
]

messages_next = [
    SystemMessage(content="당신은 주식 투자 전문가로 고객의 투자 결정에 도움을 줍니다."),
    HumanMessage(content="그렇다면 목표 주가는 얼마로 보시나요?")
]

next_human_message = HumanMessage(content="그렇다면 목표 주가는 얼마로 보시나요?")

In [13]:
response = llm.invoke(messages)
print(response.content)


print("-"*120)

# 후속 질문 (이전 맥락 없음)
response_next = llm.invoke(messages_next)
print(response_next.content)

삼성전자 주식을 매수할지 여부는 여러 요소를 고려해야 합니다. 다음은 몇 가지 면밀히 살펴볼 사항입니다.

1. **재무 상태**: 삼성전자의 최근 재무 제표와 실적 발표를 검토하여 수익성, 매출 성장률, 부채 비율 등을 확인하세요.

2. **산업 동향**: 반도체 등 삼성전자가 주로 활동하는 산업의 현재 동향과 전망을 확인해야 합니다. 반도체 시장은 특히 경기 변화에 민감하므로 수요 예측이 중요합니다.

3. **주가 평가**: 현재 주가가 역사적 평균이나 동종 업종 대비 어떻게 평가되고 있는지 분석해보세요. P/E 비율, P/B 비율 등을 활용해볼 수 있습니다.

4. **시장 분위기**: 글로벌 경제와 한국 경제의 전반적인 분위기를 살펴보세요. 경제 지표, 금리, 물가 상승률 등이 투자 결정에 영향을 미칠 수 있습니다.

5. **투자 목표 및 기간**: 개인의 투자 목표와 기간에 따라 매수 전략이 달라질 수 있습니다. 단기 투자와 장기 투자 시점에서의 판단이 다를 수 있습니다.

위의 요소들을 종합적으로 고려한 후 결정하는 것이 바람직합니다. 필요하다면 전문적인 금융 자문가와 상담하는 것도 좋은 방법입니다.
------------------------------------------------------------------------------------------------------------------------
주식의 목표 주가는 기업의 재무상태, 성장 가능성, 산업 동향, 경제 전반의 상황 등 다양한 요소에 따라 달라집니다. 특정 주식에 대한 분석을 원하신다면 그 주식의 이름이나 심볼을 말씀해 주시면, 추세, 재무 지표, 경쟁업체 분석 등을 바탕으로 보다 구체적인 의견을 드릴 수 있습니다.


In [21]:
# 1. 첫번째 질문
response = llm.invoke(messages)

# 2. 첫번째 응답 추가
messages.append(response)  # AI 응답을 히스토리에 추가

# print("-"*120)

# 3. 후속 질문 (이전 맥락 유지)
messages.append(next_human_message)
response = llm.invoke(messages)
messages.append(response)

for message in messages:
    print(type(message).__name__, ":", message.content)

SystemMessage : 당신은 주식 투자 전문가로 고객의 투자 결정에 도움을 줍니다.
HumanMessage : 삼성전자 주식을 지금 매수해도 될까요?
AIMessage : 삼성전자 주식 매수에 대한 결정은 여러 요인을 고려해야 합니다. 다음은 고려해야 할 몇 가지 측면입니다:

1. **재무 실적**: 삼성전자의 최근 재무 성과를 분석해 보세요. 매출, 순이익, 부채 비율 등 주요 지표들이 긍정적인지 확인해 보세요.

2. **산업 동향**: 반도체, 스마트폰, 가전제품 등 삼성전자가 활동하는 산업의 전반적인 시장 전망도 중요합니다. 경쟁력 있는 기술과 시장 점유율을 유지하고 있는지 확인하세요.

3. **경제 상황**: 글로벌 경제 상황, 금리 변화, 환율 변동 등이 삼성전자 주가에 영향을 미칠 수 있습니다. 특히 반도체 시장의 변화는 주가에 큰 영향을 줄 수 있습니다.

4. **기술적 분석**: 최근 주가 흐름을 분석하고, 주요 지지선과 저항선을 확인하여 매수 시점을 판단해 보세요.

5. **리스크 관리**: 투자 금액과 향후 주가 흐름에 대한 본인의 리스크 감수 성향을 고려하세요. 장기 보유를 할 것인지 단기 매매를 할 것인지 결정해야 합니다.

결정하시기 전에 이러한 요소들을 충분히 고려하시고, 필요하다면 추가 전문가의 조언을 받는 것도 좋은 방법입니다.
HumanMessage : 그렇다면 목표 주가는 얼마로 보시나요?
AIMessage : 삼성전자의 목표 주가는 다양한 분석가들의 의견에 따라 달라질 수 있으며, 정확한 목표 주가는 여러 요소에 따라 변동합니다. 다음은 목표 주가를 설정할 때 고려해야 할 주요 요소들입니다:

1. **실적 전망**: 향후 분기와 연도의 실적 예상, 매출 성장률, 순이익 증가율 등이 목표 주가에 큰 영향을 미칩니다.

2. **시장 조건**: 글로벌 경제 상황 및 산업 동향에 따라 목표 주가가 달라질 수 있습니다. 반도체 수요, 경쟁사와의 비교, 기술 발전이 큰 영향을 미칠 수 있습니다.

3. **밸류에이션*

In [24]:
from langchain_community.callbacks import get_openai_callback

with get_openai_callback() as callback:
    
    # 1. 첫번째 질문
    response = llm.invoke(messages)

    # 2. 첫번째 응답 추가
    messages.append(response)  # AI 응답을 히스토리에 추가

    # 3. 후속 질문 (이전 맥락 유지)
    messages.append(next_human_message)
    response = llm.invoke(messages)
    messages.append(response)

    for message in messages:
        print(type(message).__name__, ":", message.content[:50])
        
    print(f"\n총 토큰: {callback.total_tokens}")
    print(f"프롬프트 토큰: {callback.prompt_tokens}")
    print(f"응답 토큰: {callback.completion_tokens}")
    print(f"비용(USD): ${callback.total_cost:.6f}")


SystemMessage : 당신은 주식 투자 전문가로 고객의 투자 결정에 도움을 줍니다.
HumanMessage : 삼성전자 주식을 지금 매수해도 될까요?
AIMessage : 삼성전자 주식을 매수할지 여부는 여러 요소에 따라 달라질 수 있습니다. 다음은 고려해야 할
HumanMessage : 그렇다면 목표 주가는 얼마로 보시나요?
AIMessage : 삼성전자의 목표 주가는 여러 요소에 따라 달라질 수 있으며, 주식 분석 전문가들이 각기 다
AIMessage : 삼성전자의 목표 주가는 여러 요소에 따라 달라질 수 있으며, 주식 분석 전문가들이 각기 다
HumanMessage : 그렇다면 목표 주가는 얼마로 보시나요?
AIMessage : 삼성전자의 목표 주가는 시장 상황, 기업 성과, 산업 동향 등에 따라 지속적으로 변동하므로

총 토큰: 2384
프롬프트 토큰: 1791
응답 토큰: 593
비용(USD): $0.000548


# RunnableWithMessageHistory 

## 1. Chain 만들기

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 주식 투자 전문가입니다."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llm

## 2. 세션별 히스토리 저장소 & 이력관리체인

In [28]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

## 3. 첫 번째 질문

In [34]:
# 첫 번째 질문

with get_openai_callback() as callback:
    response = chain_with_history.invoke(
        {"input": "삼성전자 주식을 지금 매수해도 될까요?"},
        config={"configurable": {"session_id": "user_001"}}
    )
    print(response.content[:100])
    print(f"\n총 토큰: {callback.total_tokens}")
    print(f"프롬프트 토큰: {callback.prompt_tokens}")
    print(f"응답 토큰: {callback.completion_tokens}")
    print(f"비용(USD): ${callback.total_cost:.6f}")

삼성전자 주식을 지금 매수할지에 대한 결정은 개인의 투자 전략, 리스크 선호도, 시장 상황 등에 따라 다릅니다. 다음은 매수 결정을 내릴 때 고려해야 할 주요 요소들입니다:

1.

총 토큰: 1717
프롬프트 토큰: 1343
응답 토큰: 374
비용(USD): $0.000330


## 4. 후속 질문 (맥락 유지 확인)

In [35]:
# 후속 질문 - 같은 session_id로 맥락 유지
with get_openai_callback() as callback:
    
    response = chain_with_history.invoke(
        {"input": "그렇다면 목표 주가는 얼마로 보시나요?"},
        config={"configurable": {"session_id": "user_001"}}
    )
    print(response.content[:100])
    print(f"\n총 토큰: {callback.total_tokens}")
    print(f"프롬프트 토큰: {callback.prompt_tokens}")
    print(f"응답 토큰: {callback.completion_tokens}")
    print(f"비용(USD): ${callback.total_cost:.6f}")


삼성전자의 목표 주가는 여러 요인에 따라 달라질 수 있으며, 다양한 애널리스트들이 각기 다른 기준으로 목표 주가를 설정합니다. 저는 특정 목표 주가를 제시할 수는 없지만, 목표 주

총 토큰: 2085
프롬프트 토큰: 1738
응답 토큰: 347
비용(USD): $0.000469


In [33]:
# 후속 질문 - 다른 session_id로 맥락 변경
response = chain_with_history.invoke(
    {"input": "그렇다면 목표 주가는 얼마로 보시나요?"},
    config={"configurable": {"session_id": "user_002"}}
)
print(response.content[:100])

목표 주가는 특정 종목의 현재 주가, 기업의 재무 상태, 산업 동향, 기술적 분석, 경제 상황 등을 종합적으로 고려하여 설정됩니다. 특정 종목에 대한 목표 주가를 제시하기 위해서는
